# OTIF MTD — Audit & Thống kê (Mondelez)

Notebook đánh giá **`analytics_workspace.mv_otif`** trước/độc lập với session UAT (Mode B của `/da-uat`).

**Nội dung**
- **Summary sức khỏe dữ liệu** (scale · window · freshness · column coverage)
- **Thống kê** (KPI OTIF/Ontime/Infull canonical theo registry · phân bố theo dim · volume · fail-reason · trend ngày)
- **Bất thường** (5 nhóm anomaly: NULL · volume integrity · business-rule cross-field · key uniqueness · timestamp ordering)
- **Drill listout** từng nhóm vi phạm + **tra cứu 1 đơn (SO)**

**Tham số (cell L0)**
- Mặc định **MTD** (Month-To-Date: ngày 1 → hôm nay).
- Cho phép **lọc theo ngày** (đổi `MTD_FROM` / `MTD_TO`) và **lọc theo đơn** (`SO_FILTER`).
- **Góc nhìn theo ngày = `Ngày gửi thầu`** → cột `thoi_gian_gui_thau` (đúng date_type canonical trong sql-registry §OTIF).
- Đọc trực tiếp từ MV: **không áp filter dim** (kho/khu vực/NVT/loại hàng) — chỉ window ngày + (optional) SO. Muốn soi 1 dim thì dùng cell L10 ad-hoc.

> Canonical: định nghĩa Ontime/Infull/OTIF + KPI bám `sql-registry.md §OTIF` và DDL `mv_otif`. KHÔNG tự định nghĩa lại metric.

## Setup — chạy 1 lần

In [ ]:
# Setup — dùng thư viện dùng chung `da` (cài 1 lần: pip install -e ./da). Xem da/README.md.
import da
import pandas as pd
from datetime import date, timedelta
from IPython.display import display, Markdown

cfg    = da.load_tenant("mondelez")    # .env (secret) + mondelez/da.toml (config nghiệp vụ)
client = da.ch_client(cfg)
da.setup_display()                     # pd.set_option dùng chung (thay 3 dòng copy-paste)

DB = cfg.database
T  = cfg.table("mv_otif")              # `analytics_workspace`.`mv_otif`

# q() — render placeholder {key} bằng PARAMS + T + extras (GIỮ NGUYÊN convention các cell L1+).
# SQL chỉ chứa placeholder, KHÔNG chứa { } literal. Filter chuẩn: {date_col} window + {where_so}.
def q(sql: str, **extras) -> pd.DataFrame:
    return da.run_df(client, sql.format(**{**PARAMS, 'T': T, **extras}))

PARAMS = {}  # override ở cell L0
print('Setup OK — chuyển sang cell L0 để chỉnh tham số')


## L0 · Tham số — sửa ở đây rồi chạy

- **Window**: mặc định MTD. Lọc theo ngày khác → sửa `MTD_FROM` / `MTD_TO` (YYYY-MM-DD, **inclusive 2 đầu**).
- **Date type**: `DATE_COL` = cột cắt ngày. Mặc định `thoi_gian_gui_thau` (= "Ngày gửi thầu"). Các lựa chọn canonical khác ở dict `DATE_TYPES`.
- **Lọc theo đơn**: điền `SO_FILTER` = list mã SO (vd `['8482509466','8482485892']`) để chỉ soi vài đơn; để `[]` = tất cả.

In [46]:
_today = date.today()
MTD_FROM = _today.replace(day=1).isoformat()   # đầu tháng
MTD_TO   = _today.isoformat()                   # hôm nay

# Date type canonical (theo sql-registry §OTIF — CASE date filter)
DATE_TYPES = {
    'Ngày gửi thầu':        'thoi_gian_gui_thau',      # ← mặc định (user yêu cầu)
    'ETA gửi thầu (đơn)':   'eta_giao_hang_cho_npp',
    'ATA chi tiết chuyến':  'ata_den',
    'Ngày vào kho':         'gio_dang_tai',
    'Ngày duyệt chuyến':    'ngay_duyet_chuyen',
    'Ngày GI':              'ngay_gi',
    'Ngày tạo đơn hàng':    'ngay_tao_don',
}
DATE_LABEL = 'Ngày gửi thầu'
DATE_COL   = DATE_TYPES[DATE_LABEL]

UOM_COL = 'sum_original_cse'   # đơn vị volume mặc định cho thống kê (CSE)

SO_FILTER = []   # [] = tất cả đơn; hoặc ['8482509466', ...] để lọc theo đơn
_where_so = ("AND so IN (" + ",".join("'" + s.replace("'", "''") + "'" for s in SO_FILTER) + ")") if SO_FILTER else ""

PARAMS.update(
    from_date=MTD_FROM,
    to_date=MTD_TO,
    date_col=DATE_COL,
    date_label=DATE_LABEL,
    uom_col=UOM_COL,
    where_so=_where_so,
)
print(f"Window: {MTD_FROM} -> {MTD_TO}  (inclusive)")
print(f"Date type: {DATE_LABEL}  ->  column {DATE_COL}")
print(f"Volume column: {UOM_COL}")
print(f"SO filter: {SO_FILTER if SO_FILTER else 'ALL'}")

Window: 2026-05-01 -> 2026-05-29  (inclusive)
Date type: Ngày gửi thầu  ->  column thoi_gian_gui_thau
Volume column: sum_original_cse
SO filter: ALL


## L1 · Sức khỏe dữ liệu

Scale · distinct · freshness. `mv_otif` grain = `so × whseid` (ORDER BY), refresh mỗi 5 phút.

In [47]:
df_rows = q("""
SELECT count() AS rows_total_mv, uniqExact(so) AS distinct_so_total_mv
FROM {T}
""")
display(Markdown('**A · Tổng row toàn MV (không lọc window)**'))
display(df_rows)

df_distinct = q("""
SELECT
  '{date_label}'                                  AS date_type,
  '{from_date}'                                   AS from_date,
  '{to_date}'                                     AS to_date,
  count()                                         AS rows_window,
  uniqExact(so)                                   AS distinct_so,
  count() - uniqExact(so)                         AS rows_minus_so,
  uniqExact(so, whseid)                           AS distinct_so_whseid,
  uniqExact(whseid)                               AS distinct_whseid,
  uniqExact(coalesce(customer_code,''))           AS distinct_customer,
  uniqExact(coalesce(group_of_cago,''))           AS distinct_cargo,
  uniqExact(coalesce(group_name,''))              AS distinct_kenh,
  uniqExact(coalesce(khu_vuc_doi_xe,''))          AS distinct_khu_vuc,
  uniqExact(coalesce(ten_ngan_nha_van_tai,''))    AS distinct_nvt,
  countIf(otif_status = 'Không có dữ liệu STM')   AS rows_khong_co_stm,
  countIf(otif_status != 'Không có dữ liệu STM')  AS rows_co_stm
FROM {T}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
  {where_so}
""")
display(Markdown('**B · Distinct + scale trong window** (rows_minus_so > 0 = có SO split sang nhiều kho)'))
display(df_distinct.T.rename(columns={0: 'value'}))

df_fresh = q("""
SELECT
  toDateTime(max({date_col}),    'Asia/Ho_Chi_Minh') AS max_date_col,
  toDateTime(min({date_col}),    'Asia/Ho_Chi_Minh') AS min_date_col,
  toDateTime(max(ata_den),       'Asia/Ho_Chi_Minh') AS max_ata_den,
  toDateTime(max(ngay_tao_don),  'Asia/Ho_Chi_Minh') AS max_ngay_tao_don,
  now('Asia/Ho_Chi_Minh')                            AS server_now,
  dateDiff('minute', max(ata_den), now())            AS lag_min_ata_den
FROM {T}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
  {where_so}
""")
display(Markdown('**C · Freshness — max timestamp + lag vs now (UTC+7).** MV refresh 5 phút → lag lớn = MV chưa cập nhật, dừng đối chiếu.'))
display(df_fresh.T.rename(columns={0: 'value'}))

df_cov = q("""
SELECT
  round(100.0*countIf(thoi_gian_gui_thau    IS NOT NULL)/count(),1) AS pct_thoi_gian_gui_thau,
  round(100.0*countIf(etd_chuyen_gui_thau   IS NOT NULL)/count(),1) AS pct_etd,
  round(100.0*countIf(eta_giao_hang_cho_npp IS NOT NULL)/count(),1) AS pct_eta,
  round(100.0*countIf(ata_den               IS NOT NULL)/count(),1) AS pct_ata_den,
  round(100.0*countIf(actual_ship_date      IS NOT NULL)/count(),1) AS pct_actual_ship,
  round(100.0*countIf(ngay_gi               IS NOT NULL)/count(),1) AS pct_ngay_gi,
  round(100.0*countIf(ontime_status IS NOT NULL AND ontime_status!='')/count(),1) AS pct_ontime_status,
  round(100.0*countIf(infull_status IS NOT NULL AND infull_status!='')/count(),1) AS pct_infull_status,
  round(100.0*countIf(toFloat64(coalesce(sum_original_cse,0))!=0)/count(),1)      AS pct_original_cse_nonzero
FROM {T}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
  {where_so}
""")
display(Markdown('**D · Column coverage — % non-null các cột then chốt** (cột thấp = không tin dùng được trong reconciliation)'))
display(df_cov.T.rename(columns={0: 'pct_non_null'}))

**A · Tổng row toàn MV (không lọc window)**

,rows_total_mv,distinct_so_total_mv
0,1299992,1299990


**B · Distinct + scale trong window** (rows_minus_so > 0 = có SO split sang nhiều kho)

,value
date_type,Ngày gửi thầu
from_date,2026-05-01
to_date,2026-05-29
rows_window,21868
distinct_so,21868
rows_minus_so,0
distinct_so_whseid,21868
distinct_whseid,3
distinct_customer,1926
distinct_cargo,4


**C · Freshness — max timestamp + lag vs now (UTC+7).** MV refresh 5 phút → lag lớn = MV chưa cập nhật, dừng đối chiếu.

,value
max_date_col,2026-05-29 02:07:30+07:00
min_date_col,2026-05-01 17:46:08+07:00
max_ata_den,2026-05-29 18:48:09+07:00
max_ngay_tao_don,2026-05-29 00:11:14+07:00
server_now,2026-05-29 14:15:26+07:00
lag_min_ata_den,-273


**D · Column coverage — % non-null các cột then chốt** (cột thấp = không tin dùng được trong reconciliation)

,pct_non_null
pct_thoi_gian_gui_thau,100.00
pct_etd,100.00
pct_eta,100.00
pct_ata_den,100.00
pct_actual_ship,100.00
pct_ngay_gi,61.30
pct_ontime_status,100.00
pct_infull_status,100.00
pct_original_cse_nonzero,100.00


## L2 · Phân bố (distribution)

In [48]:
df_status = q("""
SELECT
  otif_status,
  count()                                        AS rows,
  round(100.0*count()/sum(count()) OVER (), 2)   AS pct,
  round(sum(toFloat64(coalesce({uom_col},0))),2) AS vol_cse
FROM {T}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
  {where_so}
GROUP BY otif_status ORDER BY rows DESC
""")
display(Markdown('**A · Phân bố `otif_status`** (gồm cả bucket `Không có dữ liệu STM` — dashboard loại bucket này khi tính %)'))
display(df_status)

for col, label in [
    ('ontime_status', 'ontime_status'),
    ('infull_status', 'infull_status'),
]:
    d = q("""
    SELECT coalesce({c},'(NULL)') AS bucket, count() AS rows,
           round(100.0*count()/sum(count()) OVER (),2) AS pct
    FROM {T}
    WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}') {where_so}
    GROUP BY bucket ORDER BY rows DESC
    """, c=col)
    display(Markdown(f'**Phân bố `{label}`**'))
    display(d)

**A · Phân bố `otif_status`** (gồm cả bucket `Không có dữ liệu STM` — dashboard loại bucket này khi tính %)

,otif_status,rows,pct,vol_cse
0,OTIF,20042,91.65,"1,001,962.03"
1,Failed OTIF,1826,8.35,"144,483.54"


**Phân bố `ontime_status`**

,bucket,rows,pct
0,Ontime,20252,92.61
1,Failed Ontime,1615,7.39
2,(NULL),1,0.00


**Phân bố `infull_status`**

,bucket,rows,pct
0,Infull,21630,98.91
1,Failed Infull,238,1.09


In [49]:
# Phân bố theo dim nghiệp vụ — bucket (NULL)/(rỗng) cao = red flag master data
for col, label in [
    ('whseid',               'Kho (whseid)'),
    ('khu_vuc_doi_xe',       'Khu vực đội xe'),
    ('group_name',           'Kênh bán hàng'),
    ('ten_ngan_nha_van_tai', 'Nhà vận tải'),
    ('group_of_cago',        'Loại hàng (Group of Cargo)'),
    ('loai_xe_gui_thau',     'Loại xe gửi thầu'),
]:
    d = q("""
    SELECT
      multiIf({c} IS NULL, '(NULL)', {c}='', '(rỗng)', {c}) AS bucket,
      count()                                       AS rows,
      round(100.0*count()/sum(count()) OVER (),2)   AS pct,
      countIf(otif_status='OTIF')                   AS otif_so,
      round(100.0*countIf(otif_status='OTIF')
            /nullIf(countIf(otif_status!='Không có dữ liệu STM'),0),2) AS pct_otif
    FROM {T}
    WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}') {where_so}
    GROUP BY bucket ORDER BY rows DESC
    """, c=col)
    display(Markdown(f'**Phân bố theo {label}** (pct_otif = %OTIF của nhóm, đã loại STM-missing ở mẫu số)'))
    display(d)

**Phân bố theo Kho (whseid)** (pct_otif = %OTIF của nhóm, đã loại STM-missing ở mẫu số)

,bucket,rows,pct,otif_so,pct_otif
0,NKD,11909,54.46,11484,96.43
1,BKD1,9713,44.42,8335,85.81
2,VN831,246,1.12,223,90.65


**Phân bố theo Khu vực đội xe** (pct_otif = %OTIF của nhóm, đã loại STM-missing ở mẫu số)

,bucket,rows,pct,otif_so,pct_otif
0,(rỗng),13461,61.56,12581,93.46
1,Ha Noi,1297,5.93,1193,91.98
2,North East - North West,1290,5.90,1139,88.29
3,Ho Chi Minh,1285,5.88,1116,86.85
4,South East,1202,5.50,1080,89.85
5,North Central Coast,966,4.42,795,82.30
6,Mekong 2,595,2.72,568,95.46
7,Central,505,2.31,468,92.67
8,Mekong 1,489,2.24,432,88.34
9,South Central Coast,361,1.65,322,89.20


**Phân bố theo Kênh bán hàng** (pct_otif = %OTIF của nhóm, đã loại STM-missing ở mẫu số)

...

,bucket,rows,pct,otif_so,pct_otif
0,HDA,10331,47.24,10041,97.19
1,ANH SON,5949,27.20,5194,87.31
2,HOA PHAT,2174,9.94,1672,76.91
3,GHN,1072,4.90,1072,100.00
4,TLL,752,3.44,594,78.99
5,NGUYEN PHAT,596,2.73,569,95.47
6,HVP,505,2.31,468,92.67
7,THANH AN,489,2.24,432,88.34


**Phân bố theo Loại hàng (Group of Cargo)** (pct_otif = %OTIF của nhóm, đã loại STM-missing ở mẫu số)

,bucket,rows,pct,otif_so,pct_otif
0,DRY,17958,82.12,16841,93.78
1,FRESH,3301,15.10,2625,79.52
2,POSM/OFFBOM,603,2.76,570,94.53
3,(NULL),6,0.03,6,100.00


**Phân bố theo Loại xe gửi thầu** (pct_otif = %OTIF của nhóm, đã loại STM-missing ở mẫu số)

,bucket,rows,pct,otif_so,pct_otif
0,2T,5211,23.83,4517,86.68
1,3.5T,3660,16.74,3522,96.23
2,(rỗng),3482,15.92,3019,86.70
3,5T,2658,12.15,2510,94.43
4,2.5T,2389,10.92,2256,94.43
5,8T,2248,10.28,2141,95.24
6,1.4T,1739,7.95,1671,96.09
7,11T,481,2.20,406,84.41


## L3 · KPI OTIF canonical (theo sql-registry §OTIF)

Định nghĩa khớp registry: `total_so = uniqExact(so)`; `%` loại bucket `Không có dữ liệu STM` ở mẫu số (`count(so)`).

In [50]:
df_kpi = q("""
WITH filtered AS (
  SELECT * FROM {T}
  WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
    {where_so}
    AND otif_status != 'Không có dữ liệu STM'   -- exclusion canonical cho % metric
)
SELECT
  uniqExact(so)                                                          AS total_so,
  countIf(ontime_status = 'Ontime')                                      AS ontime_so,
  round(100.0*countIf(ontime_status='Ontime')/nullIf(count(so),0),2)     AS pct_ontime,
  countIf(infull_status = 'Infull')                                      AS infull_so,
  round(100.0*countIf(infull_status='Infull')/nullIf(count(so),0),2)     AS pct_infull,
  countIf(otif_status = 'OTIF')                                          AS otif_so,
  round(100.0*countIf(otif_status='OTIF')/nullIf(count(so),0),2)         AS pct_otif
FROM filtered
""")
k = df_kpi.iloc[0]

# RAG band (PRD §13.2): OTIF 90 / Ontime 95 / Infull 97
def _rag(v, green, yellow_lo):
    if v is None: return '—'
    v = float(v)
    return '🟢' if v >= green else ('🟡' if v >= yellow_lo else '🔴')

display(Markdown(
    f"**Window** `{PARAMS['from_date']}` → `{PARAMS['to_date']}`  ·  **theo** `{PARAMS['date_label']}`\n\n"
    f"| KPI | Số đơn | % | Target | RAG |\n|---|---:|---:|---:|:--:|\n"
    f"| **% OTIF**   | {int(k['otif_so']):,}   | **{k['pct_otif']:.2f}%**   | 90% | {_rag(k['pct_otif'],90,85)} |\n"
    f"| **% Ontime** | {int(k['ontime_so']):,} | **{k['pct_ontime']:.2f}%** | 95% | {_rag(k['pct_ontime'],95,90)} |\n"
    f"| **% Infull** | {int(k['infull_so']):,} | **{k['pct_infull']:.2f}%** | 97% | {_rag(k['pct_infull'],97,92)} |\n\n"
    f"Tổng đơn (uniqExact SO, đã loại STM-missing): **{int(k['total_so']):,}**"
))
display(df_kpi.T.rename(columns={0:'value'}))

**Window** `2026-05-01` → `2026-05-29`  ·  **theo** `Ngày gửi thầu`

| KPI | Số đơn | % | Target | RAG |
|---|---:|---:|---:|:--:|
| **% OTIF**   | 20,042   | **91.65%**   | 90% | 🟢 |
| **% Ontime** | 20,252 | **92.61%** | 95% | 🟡 |
| **% Infull** | 21,630 | **98.91%** | 97% | 🟢 |

Tổng đơn (uniqExact SO, đã loại STM-missing): **21,868**

,value
total_so,"21,868.00"
ontime_so,"20,252.00"
pct_ontime,92.61
infull_so,"21,630.00"
pct_infull,98.91
otif_so,"20,042.00"
pct_otif,91.65


## L3b · Tổng volume MTD (Plan / Shipped / Delivered × CSE / KG / CBM)

In [51]:
df_vol = q("""
SELECT
  round(sum(toFloat64(coalesce(sum_original_cse,0))),2)        AS plan_cse,
  round(sum(toFloat64(coalesce(sum_shipped_cse,0))),2)         AS shipped_cse,
  round(sum(toFloat64(coalesce(sum_san_luong_giao_cse,0))),2)  AS delivered_cse,
  round(sum(toFloat64(coalesce(sum_original_kg,0))),2)         AS plan_kg,
  round(sum(toFloat64(coalesce(sum_shipped_kg,0))),2)          AS shipped_kg,
  round(sum(toFloat64(coalesce(sum_san_luong_giao_kg,0))),2)   AS delivered_kg,
  round(sum(toFloat64(coalesce(sum_original_cbm,0))),3)        AS plan_cbm,
  round(sum(toFloat64(coalesce(sum_shipped_cbm,0))),3)         AS shipped_cbm,
  round(sum(toFloat64(coalesce(sum_san_luong_giao_cbm,0))),3)  AS delivered_cbm
FROM {T}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}') {where_so}
  AND otif_status != 'Không có dữ liệu STM'
""")
display(Markdown('**Tổng volume** (kỳ vọng monotonic: Plan ≥ Shipped ≥ Delivered cho mỗi UOM)'))
display(df_vol.T.rename(columns={0:'value'}))

**Tổng volume** (kỳ vọng monotonic: Plan ≥ Shipped ≥ Delivered cho mỗi UOM)

,value
plan_cse,"1,146,445.57"
shipped_cse,"1,149,647.70"
delivered_cse,"1,143,299.30"
plan_kg,"4,247,572.28"
shipped_kg,"4,257,938.10"
delivered_kg,"4,235,923.56"
plan_cbm,"40,658.01"
shipped_cbm,"40,714.53"
delivered_cbm,"38,862.08"


## L3c · Trend theo ngày (góc nhìn `Ngày gửi thầu`)

Mỗi dòng = 1 ngày theo `{date_col}` (mặc định `thoi_gian_gui_thau`). Đổi `DATE_LABEL` ở L0 để xem trục ngày khác.

**Khớp dashboard — 3 điểm bám đúng widget trend (registry §"OTIF/Ontime/Infull theo thời gian"):**
1. **Cắt ngày theo UTC** — `toDate({date_col})` raw, KHÔNG `toTimeZone` UTC+7. Dashboard cắt ngày theo UTC (`DateTime64(3,'UTC')`); convert UTC+7 sẽ đẩy đơn gửi thầu buổi tối sang ngày khác → lệch số/ngày.
2. **`total_so = count(so)`** (đếm row, grain `so×whseid`) = đúng "Số đơn/day" của widget trend. Cột `so_unique` (uniqExact) cho thấy mức double-count nếu SO tách nhiều kho (`dup = count − unique`).
3. **KHÔNG loại STM-missing** ở mẫu số — widget trend tính `%` trên `count(so)` gồm cả đơn chưa có STM (khác hero KPI L3 vốn loại STM + uniqExact). Cột `rows_nostm` để thấy ảnh hưởng.

> Lưu ý: L3c reconcile với **widget Trend**; L3 reconcile với **hero KPI**. Hai widget dashboard này định nghĩa khác nhau (count vs uniqExact, có/không loại STM) — đây là đặc tính dashboard, không phải bug notebook.

In [52]:
df_trend = q("""
SELECT
  toDate({date_col})                                                     AS ngay,
  count(so)                                                              AS total_so,
  uniqExact(so)                                                          AS so_unique,
  count(so) - uniqExact(so)                                              AS dup,
  countIf(otif_status='Không có dữ liệu STM')                            AS rows_nostm,
  countIf(otif_status='OTIF')                                            AS otif_so,
  round(100.0*countIf(otif_status='OTIF')/nullIf(count(so),0),2)         AS pct_otif,
  round(100.0*countIf(ontime_status='Ontime')/nullIf(count(so),0),2)     AS pct_ontime,
  round(100.0*countIf(infull_status='Infull')/nullIf(count(so),0),2)     AS pct_infull,
  round(sum(toFloat64(coalesce(sum_original_cse,0))),2)                  AS plan_cse
FROM {T}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}') {where_so}
GROUP BY ngay ORDER BY ngay
""")
display(Markdown(f"**Trend daily theo `{PARAMS['date_label']}`** — `total_so = count(so)` cắt ngày UTC (khớp widget trend dashboard)"))
display(df_trend)

**Trend daily theo `Ngày gửi thầu`** — `total_so = count(so)` cắt ngày UTC (khớp widget trend dashboard)

,ngay,total_so,so_unique,dup,rows_nostm,otif_so,pct_otif,pct_ontime,pct_infull,plan_cse
0,2026-05-01,1416,1416,0,0,1413,99.79,99.93,99.86,"6,961.96"
1,2026-05-02,288,288,0,0,253,87.85,92.71,94.44,"35,540.13"
2,2026-05-03,1020,1020,0,0,973,95.39,96.27,98.92,"21,971.52"
3,2026-05-04,745,745,0,0,475,63.76,66.58,97.05,"61,421.77"
4,2026-05-05,2367,2367,0,0,2253,95.18,97.25,97.63,"71,656.36"
5,2026-05-06,852,852,0,0,796,93.43,95.66,97.77,"62,227.60"
6,2026-05-07,645,645,0,0,561,86.98,91.32,94.88,"60,154.10"
7,2026-05-08,221,221,0,0,207,93.67,96.83,96.83,"29,954.30"
8,2026-05-09,353,353,0,0,284,80.45,89.24,89.80,"31,710.00"
9,2026-05-10,697,697,0,0,674,96.70,96.70,99.86,"19,496.68"


## L3d · Đối chiếu chênh lệch theo ngày — TMS report #25 (L2.2) vs `mv_otif` (L3c)

So **bảng trend ngày của `mv_otif` (L3c)** với **bảng KPI ngày của TMS report #25 (L2.2 trong `tms_report_25_explore.ipynb`)**, full-outer-join theo ngày + cột delta. Cell **self-contained** — query thẳng `analytics_workspace.mdlz_tms_report_25_trip_order` bằng cùng `client`, KHÔNG phụ thuộc notebook TMS. Window = `from_date`/`to_date` ở L0.

**Trục ngày chung = `Ngày gửi thầu`:** TMS `TenderedDate` ≈ `mv_otif.thoi_gian_gui_thau` (cả hai `toDate` raw, không convert TZ).

> ⚠️ **Cờ chỉ tin được ở cột SỐ ĐƠN.** OT%/IF% hai phía **khác mẫu số** nên `Δpp` chỉ mang tính tham khảo:
> - **Đơn**: TMS = `uniqExact(OrderCode)` *sau* filter chuyến (`MasterStatus ∈ {Đã hoàn thành, Đang vận chuyển}`) + đơn (`OrderStatus = 'Đã giao hàng'`); MV = `uniqExact(so)` toàn window. Lệch đơn thường do: (a) đơn `Chờ`/chưa giao chỉ có ở 1 phía, (b) MV chưa refresh ngày mới nhất, (c) `TenderedDate` rớt sang ngày liền kề do timezone.
> - **OT%/IF%**: TMS tính trên dòng `Hoàn tất`; MV (L3c) tính trên `count(so)` **gồm cả** đơn `Không có dữ liệu STM`. Muốn apples-to-apples → đối với **L3** (hero KPI: loại STM + `uniqExact`), không phải L3c.


In [53]:
# ── L3d · Đối chiếu theo ngày: TMS report #25 (L2.2) vs mv_otif (L3c) ──────────
# Self-contained: định nghĩa macro + bảng TMS ngay trong cell. Window = PARAMS (L0).
_TMS    = "`analytics_workspace`.`mdlz_tms_report_25_trip_order`"
_GRACE  = 30                                      # dung sai on-time (phút) — khớp ONTIME_GRACE_MIN của notebook TMS
_fr, _to = PARAMS['from_date'], PARAMS['to_date']
def _dt(c):  return f"parseDateTimeBestEffortOrNull(nullIf({c}, ''))"
def _num(c): return f"toFloat64OrZero({c})"

# A · TMS daily (replica L2.2) — loại OrderCode không chuẩn + filter trạng thái chuyến/đơn
_tms = client.query_df(f"""
  SELECT toDate({_dt('TenderedDate')}) AS ngay,
         uniqExact(OrderCode) AS don_tms,
         round(100*countIf({_dt('DateToCome')} <= addMinutes({_dt('ETA')}, {_GRACE}) AND DeliveryStatus='Hoàn tất')
               / nullIf(countIf(DeliveryStatus='Hoàn tất' AND {_dt('DateToCome')} IS NOT NULL AND {_dt('ETA')} IS NOT NULL), 0), 2) AS ontime_tms,
         round(100*countIf({_num('QuantityBBGN')} >= {_num('QuantityOrder')} AND DeliveryStatus='Hoàn tất' AND {_num('QuantityOrder')} > 0)
               / nullIf(countIf(DeliveryStatus='Hoàn tất' AND {_num('QuantityOrder')} > 0), 0), 2) AS infull_tms
  FROM {_TMS}
  WHERE position(OrderCode, '-') = 0
    AND MasterStatus IN ('Đã hoàn thành','Đang vận chuyển') AND OrderStatus IN ('Đã giao hàng')
    AND toDate({_dt('TenderedDate')}) BETWEEN toDate('{_fr}') AND toDate('{_to}')
  GROUP BY ngay ORDER BY ngay
""")

# B · mv_otif daily (replica L3c: uniqExact(so); % trên count(so), KHÔNG loại STM-missing)
_mv = q("""
  SELECT toDate({date_col})                                              AS ngay,
         uniqExact(so)                                                   AS don_mv,
         round(100.0*countIf(ontime_status='Ontime')/nullIf(count(so),0),2) AS ontime_mv,
         round(100.0*countIf(infull_status='Infull')/nullIf(count(so),0),2) AS infull_mv
  FROM {T}
  WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}') {where_so}
  GROUP BY ngay ORDER BY ngay
""")

# C · Full outer join theo ngày + delta
m = pd.merge(_tms, _mv, on='ngay', how='outer').sort_values('ngay').reset_index(drop=True)
for c in ['don_tms','don_mv','ontime_tms','ontime_mv','infull_tms','infull_mv']:
    m[c] = pd.to_numeric(m[c], errors='coerce')
m['d_don']       = m['don_tms'] - m['don_mv']
m['d_ontime_pp'] = (m['ontime_tms'] - m['ontime_mv']).round(2)
m['d_infull_pp'] = (m['infull_tms'] - m['infull_mv']).round(2)

def _flag(d):
    if pd.isna(d): return '⚠️ thiếu 1 phía'
    return '🟢' if d == 0 else ('🟡' if abs(d) <= 2 else '🔴')

out = pd.DataFrame({
    'Ngày':     m['ngay'].astype(str),
    'Đơn TMS':  m['don_tms'], 'Đơn MV': m['don_mv'], 'Δ đơn': m['d_don'], 'Cờ': m['d_don'].map(_flag),
    'OT% TMS':  m['ontime_tms'], 'OT% MV': m['ontime_mv'], 'Δ OT(pp)': m['d_ontime_pp'],
    'IF% TMS':  m['infull_tms'], 'IF% MV': m['infull_mv'], 'Δ IF(pp)': m['d_infull_pp'],
})
_tot = pd.DataFrame([{
    'Ngày':'TỔNG', 'Đơn TMS': m['don_tms'].sum(), 'Đơn MV': m['don_mv'].sum(),
    'Δ đơn': m['don_tms'].sum() - m['don_mv'].sum(), 'Cờ':'',
    'OT% TMS':'', 'OT% MV':'', 'Δ OT(pp)':'', 'IF% TMS':'', 'IF% MV':'', 'Δ IF(pp)':'',
}])
out = pd.concat([out, _tot], ignore_index=True)

_n_lech = int((m['d_don'].fillna(-999) != 0).sum())
display(Markdown(
    f"**Đối chiếu theo ngày — TMS #25 (L2.2) vs mv_otif (L3c)** · trục `Ngày gửi thầu` · `{_fr}` → `{_to}`  \n"
    f"Số ngày lệch số đơn (kể cả thiếu 1 phía): **{_n_lech}** / {len(m)} ngày."
))
display(out)
print("Cờ (cột số đơn): 🟢 khớp · 🟡 lệch ≤2 · 🔴 lệch >2 · ⚠️ ngày chỉ tồn tại ở 1 phía")
print("OT%/IF% KHÁC mẫu số (TMS=dòng Hoàn tất, MV=count(so) gồm STM-missing) → Δpp chỉ tham khảo; apples-to-apples xem L3.")
print("Drill ngày lệch: m.query('d_don != 0')  ·  ngày chỉ 1 phía: m[m['don_tms'].isna() | m['don_mv'].isna()]")


**Đối chiếu theo ngày — TMS #25 (L2.2) vs mv_otif (L3c)** · trục `Ngày gửi thầu` · `2026-05-01` → `2026-05-29`  
Số ngày lệch số đơn (kể cả thiếu 1 phía): **10** / 28 ngày.

,Ngày,Đơn TMS,Đơn MV,Δ đơn,Cờ,OT% TMS,OT% MV,Δ OT(pp),IF% TMS,IF% MV,Δ IF(pp)
0,2026-05-01,NaN,1416,NaN,⚠️ thiếu 1 phía,NaN,99.93,NaN,NaN,99.86,NaN
1,2026-05-02,274.00,288,-14.00,🔴,87.37,92.71,-5.34,91.23,94.44,-3.21
2,2026-05-03,"1,016.00",1020,-4.00,🔴,92.13,96.27,-4.14,98.92,98.92,0.00
3,2026-05-04,745.00,745,0.00,🟢,58.62,66.58,-7.96,96.68,97.05,-0.37
4,2026-05-05,"2,367.00",2367,0.00,🟢,93.48,97.25,-3.77,97.52,97.63,-0.11
5,2026-05-06,852.00,852,0.00,🟢,91.54,95.66,-4.12,95.30,97.77,-2.47
6,2026-05-07,646.00,645,1.00,🟡,77.93,91.32,-13.39,94.44,94.88,-0.44
7,2026-05-08,221.00,221,0.00,🟢,78.28,96.83,-18.55,96.83,96.83,0.00
8,2026-05-09,353.00,353,0.00,🟢,86.85,89.24,-2.39,100.00,89.80,10.20
9,2026-05-10,697.00,697,0.00,🟢,86.23,96.70,-10.47,100.00,99.86,0.14


Cờ (cột số đơn): 🟢 khớp · 🟡 lệch ≤2 · 🔴 lệch >2 · ⚠️ ngày chỉ tồn tại ở 1 phía
OT%/IF% KHÁC mẫu số (TMS=dòng Hoàn tất, MV=count(so) gồm STM-missing) → Δpp chỉ tham khảo; apples-to-apples xem L3.
Drill ngày lệch: m.query('d_don != 0')  ·  ngày chỉ 1 phía: m[m['don_tms'].isna() | m['don_mv'].isna()]


## L4 · Bất thường — NULL / Empty trên field critical
Field nghiệp vụ bắt buộc mà NULL = ingestion thiếu / master data chưa map. `thoi_gian_gui_thau` NULL = đơn rớt khỏi góc nhìn theo ngày.

In [54]:
df_nulls = q("""
SELECT
  count()                                                              AS rows_window,
  countIf(so IS NULL OR so='')                                         AS so_null,
  countIf(whseid='')                                                   AS whseid_empty,
  countIf(otif_status='')                                              AS otif_status_empty,
  countIf(ontime_status IS NULL)                                       AS ontime_status_null,
  countIf(infull_status IS NULL)                                       AS infull_status_null,
  countIf(thoi_gian_gui_thau IS NULL)                                  AS thoi_gian_gui_thau_null,
  countIf(eta_giao_hang_cho_npp IS NULL)                               AS eta_null,
  countIf(ata_den IS NULL)                                             AS ata_den_null,
  countIf(customer_code IS NULL OR customer_code='')                   AS customer_code_null,
  countIf(group_of_cago IS NULL OR group_of_cago='')                   AS cargo_null,
  countIf(toFloat64(coalesce(sum_original_cse,0))=0)                   AS original_cse_zero
FROM {T}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}') {where_so}
""")
display(df_nulls.T.rename(columns={0:'count'}))

,count
rows_window,21868
so_null,0
whseid_empty,0
otif_status_empty,0
ontime_status_null,1
infull_status_null,0
thoi_gian_gui_thau_null,0
eta_null,0
ata_den_null,1
customer_code_null,0


## L5 · Bất thường — Volume integrity
Kỳ vọng monotonicity Plan ≥ Shipped ≥ Delivered (so sánh rounded 4 — khớp logic infull_status trong MV). Vi phạm = over-ship / over-deliver / âm.

In [55]:
df_int = q("""
SELECT
  count()                                                              AS rows_window,
  countIf(toFloat64(coalesce(sum_original_cse,0)) < 0)                 AS neg_plan_cse,
  countIf(toFloat64(coalesce(sum_shipped_cse,0)) < 0)                  AS neg_shipped_cse,
  countIf(toFloat64(coalesce(sum_san_luong_giao_cse,0)) < 0)          AS neg_delivered_cse,
  countIf(round(toFloat64(sum_shipped_cse),4) > round(toFloat64(sum_original_cse),4)
          AND toFloat64(sum_original_cse) > 0)                         AS overship_cse,
  countIf(round(toFloat64(sum_san_luong_giao_cse),4) > round(toFloat64(sum_shipped_cse),4)
          AND toFloat64(sum_shipped_cse) > 0)                          AS overdeliver_cse,
  countIf(toFloat64(coalesce(sum_original_cse,0))>0
          AND toFloat64(coalesce(sum_original,0))=0)                   AS cse_pos_qty_zero
FROM {T}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}') {where_so}
""")
display(df_int.T.rename(columns={0:'count'}))

,count
rows_window,21868
neg_plan_cse,0
neg_shipped_cse,0
neg_delivered_cse,0
overship_cse,233
overdeliver_cse,389
cse_pos_qty_zero,0


## L6 · Bất thường — Business rule cross-field
status ↔ fact ↔ reason mâu thuẫn. **Lưu ý nghiệp vụ**: `ontime_status` dùng grace 30 phút (`dateDiff>30`), còn nhánh on-time của `otif_status` dùng so sánh thô `ETA < ATA` (grace 0) → đơn trễ 1–30 phút có thể `Ontime` nhưng `Failed OTIF`. Dòng `grace_gap` đếm đúng case này (INFO, không phải bug).

In [56]:
df_biz = q("""
SELECT
  count()                                                              AS rows_window,
  -- reason rỗng dù failed
  countIf(ontime_status='Failed Ontime' AND (not_ontime_reason IS NULL OR not_ontime_reason=''))
                                                                       AS failontime_no_reason,
  countIf(infull_status='Failed Infull' AND (not_infull_reason IS NULL OR not_infull_reason=''))
                                                                       AS failinfull_no_reason,
  -- status vs fact
  countIf(ontime_status='Ontime' AND ata_den IS NULL)                  AS ontime_but_ata_null,
  countIf(otif_status='OTIF' AND ontime_status!='Ontime')              AS otif_but_not_ontime,
  countIf(otif_status='OTIF' AND infull_status!='Infull')              AS otif_but_not_infull,
  countIf(otif_status='Không có dữ liệu STM' AND ata_den IS NOT NULL)  AS nostm_but_has_ata,
  -- grace 30' discrepancy (INFO)
  countIf(ontime_status='Ontime' AND infull_status='Infull' AND otif_status='Failed OTIF')
                                                                       AS grace_gap,
  -- date sanity
  countIf(toDate(thoi_gian_gui_thau) > today())                       AS tender_future,
  countIf(thoi_gian_gui_thau < toDateTime64('2020-01-01 00:00:00',3,'UTC')) AS tender_too_old
FROM {T}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}') {where_so}
""")
display(df_biz.T.rename(columns={0:'count'}))

,count
rows_window,21868
failontime_no_reason,0
failinfull_no_reason,2
ontime_but_ata_null,0
otif_but_not_ontime,0
otif_but_not_infull,0
nostm_but_has_ata,0
grace_gap,0
tender_future,0
tender_too_old,0


## L7 · Bất thường — Key uniqueness
Grain MV = `(so, whseid)`. Dup `(so, whseid)` kỳ vọng 0 (SharedMergeTree có thể chưa merge). `so_multi_whseid` = SO trải nhiều kho → giải thích chênh giữa `count()` và `uniqExact(so)`.

In [57]:
dup = int(q("""
SELECT count() AS c FROM (
  SELECT so, whseid, count() AS c FROM {T}
  WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}') {where_so}
  GROUP BY so, whseid HAVING count() > 1
)
""")['c'].iloc[0])

multi = int(q("""
SELECT count() AS c FROM (
  SELECT so, uniqExact(whseid) AS w FROM {T}
  WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}') {where_so}
  GROUP BY so HAVING w > 1
)
""")['c'].iloc[0])

display(pd.DataFrame([
    ('Duplicate (so, whseid)', dup, '= 0'),
    ('SO trải > 1 whseid',     multi, 'info'),
], columns=['check', 'count', 'rule']))

,check,count,rule
0,"Duplicate (so, whseid)",0,= 0
1,SO trải > 1 whseid,0,info


## L8 · Bất thường — Timestamp ordering
Mỗi cặp phải tăng dần (`a ≤ b`). Đảo ngược = STM logging lỗi / clock skew / mapping cột sai.

In [58]:
df_ts = q("""
SELECT
  count() AS rows_window,
  countIf(thoi_gian_gui_thau IS NOT NULL AND etd_chuyen_gui_thau IS NOT NULL
          AND thoi_gian_gui_thau > etd_chuyen_gui_thau)               AS tender_after_etd,
  countIf(etd_chuyen_gui_thau IS NOT NULL AND eta_giao_hang_cho_npp IS NOT NULL
          AND etd_chuyen_gui_thau > eta_giao_hang_cho_npp)            AS etd_after_eta,
  countIf(gio_vao_cong IS NOT NULL AND gio_ra_cong IS NOT NULL
          AND gio_vao_cong > gio_ra_cong)                             AS incong_after_outcong,
  countIf(gio_vao_dock IS NOT NULL AND gio_ra_dock IS NOT NULL
          AND gio_vao_dock > gio_ra_dock)                             AS indock_after_outdock,
  countIf(ata_den IS NOT NULL AND ata_roi IS NOT NULL
          AND ata_den > ata_roi)                                      AS atadel_after_ataleave,
  countIf(actual_ship_date IS NOT NULL AND ata_den IS NOT NULL
          AND actual_ship_date > ata_den)                            AS ship_after_arrival,
  countIf(gio_ra_cong IS NOT NULL AND ata_den IS NOT NULL
          AND gio_ra_cong > ata_den)                                 AS leavegate_after_arrival,
  countIf(ngay_tao_don IS NOT NULL AND ngay_gi IS NOT NULL
          AND ngay_tao_don > ngay_gi)                                AS create_after_gi
FROM {T}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}') {where_so}
""")
display(df_ts.T.rename(columns={0:'count'}))

,count
rows_window,21868
tender_after_etd,200
etd_after_eta,0
incong_after_outcong,0
indock_after_outdock,0
atadel_after_ataleave,45
ship_after_arrival,100
leavegate_after_arrival,26
create_after_gi,136


## Summary — chạy cuối, rollup L1–L8

In [59]:
m = q("""
SELECT
  count()                                                              AS rows_window,
  uniqExact(so)                                                        AS distinct_so,
  countIf(otif_status='Không có dữ liệu STM')                          AS rows_nostm,
  toDateTime(max(ata_den),'Asia/Ho_Chi_Minh')                         AS max_ata_den,
  dateDiff('minute', max(ata_den), now())                              AS lag_min_ata,
  -- L4
  countIf(so IS NULL OR so='')                                         AS l4_so_null,
  countIf(whseid='')                                                   AS l4_whseid_empty,
  countIf(thoi_gian_gui_thau IS NULL)                                  AS l4_tender_null,
  countIf(ontime_status IS NULL)                                       AS l4_ontime_null,
  countIf(infull_status IS NULL)                                       AS l4_infull_null,
  -- L5
  countIf(toFloat64(coalesce(sum_original_cse,0))<0)                   AS l5_neg_plan,
  countIf(round(toFloat64(sum_shipped_cse),4)>round(toFloat64(sum_original_cse),4) AND toFloat64(sum_original_cse)>0) AS l5_overship,
  countIf(round(toFloat64(sum_san_luong_giao_cse),4)>round(toFloat64(sum_shipped_cse),4) AND toFloat64(sum_shipped_cse)>0) AS l5_overdeliver,
  -- L6
  countIf(otif_status='OTIF' AND ontime_status!='Ontime')              AS l6_otif_not_ontime,
  countIf(otif_status='OTIF' AND infull_status!='Infull')              AS l6_otif_not_infull,
  countIf(ontime_status='Failed Ontime' AND (not_ontime_reason IS NULL OR not_ontime_reason='')) AS l6_failontime_noreason,
  countIf(ontime_status='Ontime' AND infull_status='Infull' AND otif_status='Failed OTIF') AS l6_grace_gap,
  countIf(toDate(thoi_gian_gui_thau)>today())                          AS l6_tender_future,
  -- L8
  countIf(gio_vao_cong IS NOT NULL AND gio_ra_cong IS NOT NULL AND gio_vao_cong>gio_ra_cong)   AS l8_cong,
  countIf(ata_den IS NOT NULL AND ata_roi IS NOT NULL AND ata_den>ata_roi)                     AS l8_ata,
  countIf(actual_ship_date IS NOT NULL AND ata_den IS NOT NULL AND actual_ship_date>ata_den)   AS l8_ship_after_arr,
  countIf(ngay_tao_don IS NOT NULL AND ngay_gi IS NOT NULL AND ngay_tao_don>ngay_gi)           AS l8_create_after_gi
FROM {T}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}') {where_so}
""").iloc[0]

dup = int(q("""
SELECT count() AS c FROM (
  SELECT so, whseid, count() AS c FROM {T}
  WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}') {where_so}
  GROUP BY so, whseid HAVING count()>1)
""")['c'].iloc[0])

_OK, _FAIL, _INFO = '✓ OK', '✗ FAIL', '—'
def _eq0(v): return _OK if int(v)==0 else _FAIL
def _gt0(v): return _OK if int(v)>0 else _FAIL
def _lt(v,t): return _OK if int(v)<t else _FAIL

rows = [
    ('L1 Health','Rows window',                  int(m['rows_window']),  _gt0(m['rows_window']),'> 0'),
    ('L1 Health','Distinct SO',                  int(m['distinct_so']),  _INFO,'info'),
    ('L1 Health','Rows không có STM',            int(m['rows_nostm']),   _INFO,'info (loại khỏi % KPI)'),
    ('L1 Health','Lag ATA (phút vs now)',        int(m['lag_min_ata']),  _INFO,'info'),
    ('L4 NULL','so NULL/empty',                  int(m['l4_so_null']),   _eq0(m['l4_so_null']),'= 0'),
    ('L4 NULL','whseid empty',                   int(m['l4_whseid_empty']),_eq0(m['l4_whseid_empty']),'= 0'),
    ('L4 NULL','thoi_gian_gui_thau NULL',        int(m['l4_tender_null']),_INFO,'info (rớt khỏi trend ngày)'),
    ('L4 NULL','ontime_status NULL',             int(m['l4_ontime_null']),_INFO,'info'),
    ('L4 NULL','infull_status NULL',             int(m['l4_infull_null']),_INFO,'info'),
    ('L5 Volume','plan_cse < 0',                 int(m['l5_neg_plan']),  _eq0(m['l5_neg_plan']),'= 0'),
    ('L5 Volume','over-ship (shipped>plan)',     int(m['l5_overship']),  _INFO,'info (→ Failed Infull)'),
    ('L5 Volume','over-deliver (giao>shipped)',  int(m['l5_overdeliver']),_eq0(m['l5_overdeliver']),'= 0'),
    ('L6 Business','OTIF nhưng !Ontime',         int(m['l6_otif_not_ontime']),_eq0(m['l6_otif_not_ontime']),'= 0'),
    ('L6 Business','OTIF nhưng !Infull',         int(m['l6_otif_not_infull']),_eq0(m['l6_otif_not_infull']),'= 0'),
    ('L6 Business','Failed Ontime thiếu reason', int(m['l6_failontime_noreason']),_eq0(m['l6_failontime_noreason']),'= 0'),
    ('L6 Business','grace-gap (Ontime+Infull nhưng Failed OTIF)', int(m['l6_grace_gap']),_INFO,'info (grace 30p)'),
    ('L6 Business','tender date tương lai',      int(m['l6_tender_future']),_INFO,'info'),
    ('L7 Key','Duplicate (so, whseid)',          dup,                    _eq0(dup),'= 0'),
    ('L8 Time','gio_vao_cong > gio_ra_cong',     int(m['l8_cong']),      _eq0(m['l8_cong']),'= 0'),
    ('L8 Time','ata_den > ata_roi',              int(m['l8_ata']),       _eq0(m['l8_ata']),'= 0'),
    ('L8 Time','actual_ship > ata_den',          int(m['l8_ship_after_arr']),_eq0(m['l8_ship_after_arr']),'= 0'),
    ('L8 Time','ngay_tao_don > ngay_gi',         int(m['l8_create_after_gi']),_eq0(m['l8_create_after_gi']),'= 0'),
]
df_summary = pd.DataFrame(rows, columns=['Section','Metric','Value','Status','Rule'])

def _color(s):
    out=[]
    for v in s:
        if v==_OK: out.append('background-color:#DCFCE7;color:#166534;font-weight:600')
        elif v==_FAIL: out.append('background-color:#FEE2E2;color:#991B1B;font-weight:600')
        else: out.append('color:#6B7280')
    return out

n_ok=int((df_summary['Status']==_OK).sum()); n_fail=int((df_summary['Status']==_FAIL).sum()); n_info=int((df_summary['Status']==_INFO).sum())

def _hide(r):
    if r['Section']=='L1 Health': return False
    if r['Status']!=_INFO: return False
    try: return int(r['Value'])==0
    except (TypeError,ValueError): return False
mask=~df_summary.apply(_hide,axis=1); df_vis=df_summary[mask]; n_hidden=len(df_summary)-len(df_vis)

display(Markdown(
    f"**Window:** `{PARAMS['from_date']}` → `{PARAMS['to_date']}`  ·  **Date:** `{PARAMS['date_col']}` ({PARAMS['date_label']})  ·  "
    f"**SO filter:** `{'ALL' if not SO_FILTER else SO_FILTER}`\n\n"
    f"**Result:** ✓ OK = {n_ok}  ·  ✗ FAIL = {n_fail}  ·  — info = {n_info}"
    + (f"  →  ⚠ **{n_fail} metric vi phạm — drill bằng L9/L10**" if n_fail else "  →  ✓ Data healthy, không có anomaly cứng")
    + (f"\n\n_({n_hidden} dòng info=0 đã ẩn — set `SHOW_ALL=True` để xem hết)_" if n_hidden else "")
))
SHOW_ALL=False
display((df_summary if SHOW_ALL else df_vis).style.apply(_color, subset=['Status']).hide(axis='index'))

**Window:** `2026-05-01` → `2026-05-29`  ·  **Date:** `thoi_gian_gui_thau` (Ngày gửi thầu)  ·  **SO filter:** `ALL`

**Result:** ✓ OK = 9  ·  ✗ FAIL = 4  ·  — info = 9  →  ⚠ **4 metric vi phạm — drill bằng L9/L10**

_(4 dòng info=0 đã ẩn — set `SHOW_ALL=True` để xem hết)_

Section,Metric,Value,Status,Rule
L1 Health,Rows window,21868,✓ OK,> 0
L1 Health,Distinct SO,21868,—,info
L1 Health,Rows không có STM,0,—,info (loại khỏi % KPI)
L1 Health,Lag ATA (phút vs now),-273,—,info
L4 NULL,so NULL/empty,0,✓ OK,= 0
L4 NULL,whseid empty,0,✓ OK,= 0
L4 NULL,ontime_status NULL,1,—,info
L5 Volume,plan_cse < 0,0,✓ OK,= 0
L5 Volume,over-ship (shipped>plan),233,—,info (→ Failed Infull)
L5 Volume,over-deliver (giao>shipped),389,✗ FAIL,= 0


## L9 · Drill listout — đơn vi phạm + tra cứu

### L9.1 · Fail reason breakdown (not_ontime_reason / not_infull_reason)

In [60]:
d1 = q("""
SELECT coalesce(nullIf(not_ontime_reason,''),'(rỗng)') AS reason, count() AS fail_so,
       round(100.0*count()/sum(count()) OVER (),2) AS pct
FROM {T}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}') {where_so}
  AND ontime_status='Failed Ontime'
GROUP BY reason ORDER BY fail_so DESC
""")
display(Markdown('**Fail Ontime — bucket `not_ontime_reason`**'))
display(d1)

d2 = q("""
SELECT coalesce(nullIf(not_infull_reason,''),'(rỗng)') AS reason, count() AS fail_so,
       round(100.0*count()/sum(count()) OVER (),2) AS pct
FROM {T}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}') {where_so}
  AND infull_status='Failed Infull'
GROUP BY reason ORDER BY fail_so DESC
""")
display(Markdown('**Fail Infull — bucket `not_infull_reason`**'))
display(d2)

**Fail Ontime — bucket `not_ontime_reason`**

,reason,fail_so,pct
0,Late arrival by Transport,870,53.87
1,Late delivery by Transport,405,25.08
2,Thiếu dữ liệu đăng ký dock,172,10.65
3,Late warehouse call by Warehouse,167,10.34
4,Late pickup by Warehouse,1,0.06


**Fail Infull — bucket `not_infull_reason`**

,reason,fail_so,pct
0,Warehouse Infull Failure,119,50.00
1,Transport Infull Failure,114,47.90
2,Warehouse + Transport Infull Failure,3,1.26
3,(rỗng),2,0.84


### L9.2 · Over-deliver — `san_luong_giao_cse > shipped_cse` (top theo độ nặng)

In [61]:
d = q("""
SELECT so, whseid, ten_ngan_nha_van_tai,
       round(toFloat64(sum_original_cse),2)       AS plan_cse,
       round(toFloat64(sum_shipped_cse),2)        AS shipped_cse,
       round(toFloat64(sum_san_luong_giao_cse),2) AS delivered_cse,
       round(toFloat64(sum_san_luong_giao_cse)-toFloat64(sum_shipped_cse),2) AS over_cse,
       infull_status
FROM {T}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}') {where_so}
  AND round(toFloat64(sum_san_luong_giao_cse),4) > round(toFloat64(sum_shipped_cse),4)
  AND toFloat64(sum_shipped_cse) > 0
ORDER BY over_cse DESC LIMIT 100
""")
display(d)

,so,whseid,ten_ngan_nha_van_tai,plan_cse,shipped_cse,delivered_cse,over_cse,infull_status
0,8482520023,BKD1,HVP,264.00,264.00,528.00,264.00,Infull
1,8482520205,BKD1,HVP,255.00,255.00,510.00,255.00,Infull
2,8482520081,BKD1,HVP,100.00,100.00,200.00,100.00,Infull
3,8482508264,VN831,HDA,212.00,212.00,308.00,96.00,Infull
4,8482520077,BKD1,HVP,78.00,78.00,156.00,78.00,Infull
5,8482510270,NKD,HDA,354.00,288.00,354.00,66.00,Failed Infull
6,8482515466,NKD,HDA,521.00,521.00,577.00,56.00,Infull
7,8482501823,NKD,HDA,16.00,16.00,67.00,51.00,Infull
8,8482500128,NKD,HDA,"1,048.00","1,000.00","1,048.00",48.00,Failed Infull
9,8482501336,NKD,HDA,170.00,122.00,170.00,48.00,Failed Infull


### L9.3 · Grace-gap — `Ontime` + `Infull` nhưng `Failed OTIF` (trễ 1–30 phút)

In [62]:
d = q("""
SELECT so, whseid, ten_ngan_nha_van_tai,
       toDateTime(eta_giao_hang_cho_npp,'Asia/Ho_Chi_Minh') AS eta,
       toDateTime(ata_den,'Asia/Ho_Chi_Minh')               AS ata,
       dateDiff('minute', eta_giao_hang_cho_npp, ata_den)   AS tre_phut,
       ontime_status, infull_status, otif_status
FROM {T}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}') {where_so}
  AND ontime_status='Ontime' AND infull_status='Infull' AND otif_status='Failed OTIF'
ORDER BY tre_phut DESC LIMIT 100
""")
display(Markdown('Nếu có dòng: xác nhận với khách định nghĩa OTIF on-time leg dùng grace 0 (ETA<ATA) hay 30 phút như Ontime.'))
display(d)

Nếu có dòng: xác nhận với khách định nghĩa OTIF on-time leg dùng grace 0 (ETA<ATA) hay 30 phút như Ontime.

""


### L9.4 · Duplicate `(so, whseid)`

In [63]:
d = q("""
SELECT so, whseid, count() AS c
FROM {T}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}') {where_so}
GROUP BY so, whseid HAVING count()>1
ORDER BY c DESC LIMIT 100
""")
display(d if len(d) else Markdown('_Không có duplicate._'))

_Không có duplicate._

### L9.5 · Timestamp reversal — `actual_ship_date > ata_den` (xuất kho sau khi đã đến)

In [64]:
d = q("""
SELECT so, whseid, khu_vuc_doi_xe,
       toDateTime(actual_ship_date,'Asia/Ho_Chi_Minh') AS actual_ship_vn,
       toDateTime(ata_den,'Asia/Ho_Chi_Minh')          AS ata_den_vn,
       dateDiff('hour', ata_den, actual_ship_date)      AS hours_reversed
FROM {T}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}') {where_so}
  AND actual_ship_date IS NOT NULL AND ata_den IS NOT NULL AND actual_ship_date > ata_den
ORDER BY hours_reversed DESC LIMIT 100
""")
display(d if len(d) else Markdown('_Không có vi phạm._'))
if len(d):
    h = q("""
    SELECT whseid, khu_vuc_doi_xe, count() AS vi_pham
    FROM {T}
    WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}') {where_so}
      AND actual_ship_date IS NOT NULL AND ata_den IS NOT NULL AND actual_ship_date > ata_den
    GROUP BY whseid, khu_vuc_doi_xe ORDER BY vi_pham DESC
    """)
    display(Markdown('**Hotspot theo kho × khu vực**'))
    display(h)

,so,whseid,khu_vuc_doi_xe,actual_ship_vn,ata_den_vn,hours_reversed
0,8482500546,NKD,North East - North West,2026-05-14 07:51:53+07:00,2026-05-11 14:36:54+07:00,65
1,8482501753,NKD,,2026-05-14 07:52:21+07:00,2026-05-11 18:56:42+07:00,61
2,8482500547,NKD,,2026-05-14 07:52:21+07:00,2026-05-11 18:56:42+07:00,61
3,8482510023,NKD,North East - North West,2026-05-20 12:44:07+07:00,2026-05-19 13:34:38+07:00,23
4,8482500690,NKD,Ha Noi,2026-05-09 16:50:12+07:00,2026-05-09 07:20:42+07:00,9
5,8482501392,NKD,Ha Noi,2026-05-09 16:50:28+07:00,2026-05-09 08:19:09+07:00,8
6,8482501391,NKD,Ha Noi,2026-05-09 16:50:28+07:00,2026-05-09 08:19:09+07:00,8
7,8482501338,NKD,Ha Noi,2026-05-09 16:50:28+07:00,2026-05-09 08:19:09+07:00,8
8,8482500689,NKD,Ha Noi,2026-05-09 16:50:28+07:00,2026-05-09 08:19:09+07:00,8
9,8482509847,NKD,Ha Noi,2026-05-18 17:17:51+07:00,2026-05-18 10:13:03+07:00,7


**Hotspot theo kho × khu vực**

,whseid,khu_vuc_doi_xe,vi_pham
0,NKD,Ha Noi,47
1,NKD,North East - North West,31
2,NKD,North Central Coast,13
3,BKD1,Ho Chi Minh,5
4,BKD1,,2
5,NKD,,2


### L9.6 · Tra cứu 1 đơn (SO lookup)
Điền `SO_LOOKUP` để dump toàn bộ record + diễn giải trạng thái của 1 đơn.

In [65]:
SO_LOOKUP = ''   # vd '8482509466'
if SO_LOOKUP:
    d = client.query_df(f"""
    SELECT so, whseid, group_of_cago, group_name, customer_name, khu_vuc_doi_xe, ten_ngan_nha_van_tai,
           toDateTime(thoi_gian_gui_thau,'Asia/Ho_Chi_Minh')    AS ngay_gui_thau,
           toDateTime(etd_chuyen_gui_thau,'Asia/Ho_Chi_Minh')   AS etd,
           toDateTime(eta_giao_hang_cho_npp,'Asia/Ho_Chi_Minh') AS eta,
           toDateTime(actual_ship_date,'Asia/Ho_Chi_Minh')      AS actual_ship_vn,
           toDateTime(ata_den,'Asia/Ho_Chi_Minh')               AS ata_den_vn,
           dateDiff('minute', eta_giao_hang_cho_npp, ata_den)   AS tre_phut,
           round(toFloat64(sum_original_cse),2)       AS plan_cse,
           round(toFloat64(sum_shipped_cse),2)        AS shipped_cse,
           round(toFloat64(sum_san_luong_giao_cse),2) AS delivered_cse,
           ontime_status, infull_status, otif_status, not_ontime_reason, not_infull_reason
    FROM {T} WHERE so = '{SO_LOOKUP}'
    """.format(T=T, SO_LOOKUP=SO_LOOKUP.replace("'", "''")))
    display(d.T.rename(columns={i: f'whseid_{i}' for i in range(len(d))}))
else:
    display(Markdown('_Điền `SO_LOOKUP` rồi chạy lại để tra cứu 1 đơn._'))

_Điền `SO_LOOKUP` rồi chạy lại để tra cứu 1 đơn._

## L10 · Ad-hoc SQL — slot truy vấn tự do
Đổi `sql` rồi chạy. Dùng `{T}` cho bảng, `{date_col}` `{from_date}` `{to_date}` `{where_so}` cho window. Muốn lọc 1 dim (kho/NVT/...) tự thêm clause ở đây (đọc trực tiếp MV không cần filter contract của dashboard).

In [66]:
sql = """
SELECT whseid, count() AS rows, uniqExact(so) AS so,
       round(100.0*countIf(otif_status='OTIF')/nullIf(countIf(otif_status!='Không có dữ liệu STM'),0),2) AS pct_otif
FROM {T}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}') {where_so}
GROUP BY whseid ORDER BY so DESC
"""
display(q(sql))

,whseid,rows,so,pct_otif
0,NKD,11909,11909,96.43
1,BKD1,9713,9713,85.81
2,VN831,246,246,90.65
